# Stationary and non-stationary methods

In [ ]:
#    APM41012EP course notebook - Chapter 5 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Iterative methods for solving linear systems
#    Stationary and non-stationary methods
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
from scipy.sparse import diags
from scipy.sparse.linalg import inv
from scipy.sparse.linalg import norm
import plotly.graph_objects as go
from plotly.subplots import make_subplots

We want to solve the elliptic problem given by the Poisson equation subject to Dirichlet boundary conditions:

$$
\left\{
\begin{aligned}
-\Delta u(x) & =  b(x) \quad \text{in} \; \Omega = [0,1] \quad \text{with} \; b(x)=1\\
        u(0) & =  0 \quad \text{on}  \;  \partial \Omega
\end{aligned}
\right.
$$

In [ ]:
# nb of points without boundaries
nx = 100
dx = 1/(nx+1)
diagonals = [np.repeat(2/(dx*dx), nx), np.repeat(-1/(dx*dx), nx-1), np.repeat(-1/(dx*dx), nx-1)]
a = diags(diagonals, [0, -1, 1])
norm_a = norm(a)

b = np.ones(nx)

## Jacobi method

In [ ]:
def jacobi(a, b, max_iter=100000, eps=1.e-6):
    xk = np.zeros(b.size)
    norm_b = np.linalg.norm(b)

    lpu = diags([a.diagonal(-1), a.diagonal(1)], [-1, 1])
    inv_d = diags([1/a.diagonal()], [0])

    ite_mat = -inv_d*lpu
    
    hist_norm_rk = []
    hist_norm_rk.append(1.0)
    
    for i in range(max_iter+1):
        xk = ite_mat.dot(xk) + inv_d.dot(b)
        rk = a.dot(xk) - b
        norm_rk = np.linalg.norm(rk) / norm_b
        hist_norm_rk.append(norm_rk)
        if (norm_rk < eps): break

    print(f"  Number of iterations = {i+1}")
    print(f"  ||A.xk - b|| / ||b||                = {norm_rk}")

    return xk, hist_norm_rk

In [ ]:
print("\nSolution using the Jacobi method")
u, hist_jac = jacobi(a, b, max_iter=30000)

nit_jac = len(hist_jac)

res = np.linalg.norm(b - a.dot(u))
print(f"  ||A.xk - b|| / ||A|| ||xk|| + ||b|| = {res / (norm_a * np.linalg.norm(u) + np.linalg.norm(b))}")

fig = go.Figure(go.Scatter(x=np.arange(nit_jac), y=hist_jac[:nit_jac]))
fig.update_xaxes(title="number of iterations")
fig.update_yaxes(type="log", exponentformat = 'e', title="||A.xk - b|| / ||b||")
fig.update_layout(title="Convergence history")
fig.show()

## Gauss-Seidel method

In [ ]:
def gauss_seidel(a, b, max_iter=100000, eps=1.e-6):
    xk = np.zeros(b.size)
    norm_b = np.linalg.norm(b)

    dpl = diags([a.diagonal(-1), a.diagonal(0)], [-1, 0])
    inv_dpl = inv(dpl.tocsc())
    u = diags([a.diagonal(1)], [1])
    ite_mat = -inv_dpl*u
    
    hist_norm_rk = []
    hist_norm_rk.append(1.0)

    for k in range(max_iter):
        xk = ite_mat.dot(xk) + inv_dpl.dot(b)
        rk = a.dot(xk) - b
        norm_rk = np.linalg.norm(rk)/norm_b
        hist_norm_rk.append(norm_rk)
        if (norm_rk < eps): break

    print(f"  Number of iterations = {k+1}")
    print(f"  ||A.xk - b|| / ||b||                = {norm_rk}")

    return xk, hist_norm_rk

In [ ]:
print("\nSolution using the Gauss-Seidel method")
u, hist_gauss = gauss_seidel(a, b, max_iter=30000)

nit_gauss = len(hist_gauss)

res = np.linalg.norm(b - a.dot(u))
print(f"  ||A.xk - b|| / ||A|| ||xk|| + ||b|| = {res / (norm_a * np.linalg.norm(u) + np.linalg.norm(b))}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nit_jac), y=hist_jac, name="Jacobi"))
fig.add_trace(go.Scatter(x=np.arange(nit_gauss), y=hist_gauss, name="Gauss-Seidel"))
fig.update_xaxes(title="number of iterations")
fig.update_yaxes(type="log", exponentformat = 'e', title="||A.xk - b|| / ||b||")
fig.update_layout(title="Convergence history")
fig.show()

## Conjugate gradient method

In [ ]:
def conjugate_gradient(a, b, tol=1e-6):
    xk = np.zeros(b.size)
    norm_b = np.linalg.norm(b)

    rk = b - a.dot(xk)
    pk = rk
    rkm1 = rk
    
    hist_norm_rk = []
    hist_norm_rk.append(1.0)

    for k in range(b.size):
        apk = a.dot(pk)
        alpha = np.dot(rk,rk) / np.dot(pk, apk)
        xk = xk + alpha*pk
        rk = rk - alpha*apk
        norm_rk = np.linalg.norm(rk)
        hist_norm_rk.append(norm_rk/norm_b)
        if norm_rk/norm_b < tol: break
        beta = np.dot(rk,rk) / np.dot(rkm1, rkm1)
        pk = rk + beta*pk
        rkm1 = rk

    #print(f"  Number of iterations = {k+1}")
    #print(f"  ||A.xk - b|| / ||b|| = {norm_rk/norm_b}")

    return xk, hist_norm_rk

def conjugate_gradient_bis(a, b, xexa, tol=1e-6):
    xk = np.zeros(b.size)
    norm_b = np.linalg.norm(b)

    rk = b - a.dot(xk)
    pk = rk
    rkm1 = rk
    
    hist_norm_rk = []
    hist_norm_rk.append(1.0)
    hist_norm_err = []
    hist_norm_err.append(1.0)
    err0 = np.dot(xexa, a.dot(xexa))

    for k in range(b.size):
        apk = a.dot(pk)
        alpha = np.dot(rk,rk) / np.dot(pk, apk)
        xk = xk + alpha*pk
        rk = rk - alpha*apk
        norm_rk = np.linalg.norm(rk)
        hist_norm_rk.append(norm_rk/norm_b)
        err = xexa - xk
        hist_norm_err.append(np.sqrt(np.dot(err, a.dot(err))/err0))
        if norm_rk/norm_b < tol: break
        beta = np.dot(rk,rk) / np.dot(rkm1, rkm1)
        pk = rk + beta*pk
        rkm1 = rk

    print(f"  Number of iterations = {k+1}")
    print(f"  ||rk|| / ||b||                      = {norm_rk/norm_b}")

    return xk, hist_norm_rk, hist_norm_err

In [ ]:
print("\nSolution using the conjugate gradient method")
u, hist_cg = conjugate_gradient(a, b)
u, hist_cg, hist_err = conjugate_gradient_bis(a, b, u)

res = np.linalg.norm(b - a.dot(u))
print(f"  ||A.xk - b|| / ||A|| ||xk|| + ||b|| = {res / (norm_a * np.linalg.norm(u) + np.linalg.norm(b))}")

nit_cg = len(hist_cg)

fig = make_subplots(rows=2, cols=1, subplot_titles=("", "Zoom on the first 50 iterations"), vertical_spacing=0.1)

fig.add_trace(go.Scatter(x=np.arange(nit_jac),   y=hist_jac,   name="Jacobi"), row=1, col=1)
fig.add_trace(go.Scatter(x=np.arange(nit_gauss), y=hist_gauss, name="Gauss-Seidel"), row=1, col=1)
fig.add_trace(go.Scatter(x=np.arange(nit_cg),    y=hist_cg,    name="Conjugate gradient"), row=1, col=1)

fig.add_trace(go.Scatter(x=np.arange(nit_cg), y=hist_jac[:nit_cg], showlegend=False, marker_color='rgb(76,114,176)'), row=2, col=1)
fig.add_trace(go.Scatter(x=np.arange(nit_cg), y=hist_gauss[:nit_cg], showlegend=False, marker_color='rgb(221,132,82)'), row=2, col=1)
fig.add_trace(go.Scatter(x=np.arange(nit_cg), y=hist_cg, showlegend=False, marker_color='rgb(85,168,104)'), row=2, col=1)

fig.update_xaxes(title="number of iterations")
fig.update_yaxes(type="log", exponentformat = 'e', title="||A.xk - b|| / ||b||")
fig.update_layout(title="Convergence history", height=1000)
              
fig.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nit_cg), y=hist_err, name="Error in A-norm"))
fig.add_trace(go.Scatter(x=np.arange(nit_cg), y=hist_cg, name="||A.xk - b|| / ||b||"))
fig.update_xaxes(title="number of iterations")
fig.update_yaxes(type="log", exponentformat = 'e')
fig.update_layout(title="Convergence history", legend = dict(orientation="h", y=1.1))
fig.show()